In [1]:
import os
import re
import requests
import pandas as pd
from dotenv import load_dotenv
from bs4 import BeautifulSoup

# 1. 환경변수 로드
load_dotenv(override=True)
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")

# 2. HTML 특수문자 및 태그 정제 함수
def clean_html(text):
    clean_text = re.sub(r'<.*?>', '', text)
    clean_text = BeautifulSoup(clean_text, "html.parser").text
    return clean_text.strip()

# 3. 특정 상권 뉴스 검색 함수
def fetch_commercial_area_news(area_name, keyword="상권", display=30):
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id": CLIENT_ID,
        "X-Naver-Client-Secret": CLIENT_SECRET,
    }
    
    # 상권 검색 쿼리 구성 (예: '이북5도청사 상권', '독립문역 상권')
    # '독립문역 1번'처럼 세부 출구 번호가 붙은 경우 검색률을 위해 정리하거나 그대로 조합 가능
    query = f"{area_name} {keyword}"
    params = {
        "query": query,
        "display": display,
        "sort": "sim"  # 관련도순
    }
    
    response = requests.get(url, headers=headers, params=params)
    
    if not response.ok:
        print(f"[{area_name}] 호출 실패: {response.status_code}")
        return []
    
    items = response.json().get("items", [])
    news_list = []
    
    # 4. 광고/분양 노이즈 필터링
    noise_keywords = ['분양', '모델하우스', '체험단', '협찬', '입주자 모집']
    
    for item in items:
        title = clean_html(item.get("title", ""))
        description = clean_html(item.get("description", ""))
        
        if any(noise in title or noise in description for noise in noise_keywords):
            continue
            
        news_list.append({
            "trdar_cd_nm": area_name,  # 상권 데이터 컬럼명과 일치시킴
            "title": title,
            "description": description,
            "originallink": item.get("originallink", ""),
            "link": item.get("link", ""),
            "pubDate": item.get("pubDate", "")
        })
        
    return news_list

# 5. 대상 상권 리스트 설정 및 일괄 수집
target_areas = ["이북5도청사", "독립문역 1번"]
all_news = []

for area in target_areas:
    print(f">> [{area}] 상권 뉴스 수집 중...")
    news_data = fetch_commercial_area_news(area, keyword="상권", display=30)
    all_news.extend(news_data)

# 6. DataFrame 변환 및 파일 저장
df_news = pd.DataFrame(all_news)

os.makedirs("data", exist_ok=True)
df_news.to_csv("data/seoul_area_news.csv", index=False, encoding="utf-8-sig")
df_news.to_json("data/seoul_area_news.json", orient="records", force_ascii=False, indent=4)

print(f"\n수집 완료! 총 {len(df_news)}건 저장됨.")
if not df_news.empty:
    print(df_news.groupby('trdar_cd_nm').head(2)[['trdar_cd_nm', 'title']])
else:
    print("수집된 뉴스 기사가 없습니다.")

>> [이북5도청사] 상권 뉴스 수집 중...
>> [독립문역 1번] 상권 뉴스 수집 중...

수집 완료! 총 15건 저장됨.
   trdar_cd_nm                                        title
0       이북5도청사           '정부세종 4청사' 건립 시기... '상권 공실' 맞물려 주목
1       이북5도청사  홍순식 세종시장 예비후보, 공약 2호 '세종 인구 5만명 순증 프로젝트'...
10     독립문역 1번                 집객력 높은 대로변 스트리트형 상가로 투자자 몰린다
11     독립문역 1번                아파트 흥행 이어간다…단지 내 상가 잇따라 완판 행진


C:\Users\user\AppData\Local\Temp\ipykernel_14860\1404904294.py:16: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  clean_text = BeautifulSoup(clean_text, "html.parser").text
